In [1]:
%run /home/hadoop/agrim_cdp/common_utils/db.ipynb

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import logging
from datetime import datetime
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, current_timestamp
from pyspark.sql.functions import to_date

In [3]:
# ---- Configurations ----
BUCKET = 'agrim-cdp'
PREFIX = 'data-landing/callyzer/'
BATCH_SIZE = 300
TABLE_NAME = 'cdp_raw_db.callyzer_call_logs_raw'
S3_REGION = 'ap-south-1'

In [4]:
# ---- Initialize Spark & boto3 ----
spark = SparkSession.builder.appName("callyzer_batch_load").config("spark.jars", "/home/hadoop/agrim_cdp/common_jars/postgresql-42.2.24.jar").getOrCreate()
s3 = boto3.client('s3', region_name=S3_REGION)
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/06/29 08:00:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/06/29 08:00:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


25/06/29 08:00:10 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [5]:
# ---- Step 1: List files ----
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=BATCH_SIZE)
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.txt')]

In [6]:
if not files:
    logger.info("No JSON files found.")
    exit(0)

In [7]:
logger.info(f"Processing {len(files)} files.")

INFO:__main__:Processing 83 files.


In [8]:
# ---- Step 2: Read files into Spark DataFrame ----
file_paths = [f"s3a://{BUCKET}/{key}" for key in files]
raw_df = spark.read.option("multiline", "true").json(file_paths)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [9]:
# ---- Step 3: Flatten nested structure ----
flattened_df = (
    raw_df
    .withColumn("log", explode("logs"))
    .select(
        col("employeeName").alias("employee_name"),
        col("employeeCode").alias("employee_code"),
        col("countryCode").alias("employee_country_code"),
        col("employeeNumber").alias("employee_number"),
        col("employeeTags").alias("employee_tags"),
        col("log.id").alias("id"),
        col("log.name").alias("name"),
        col("log.countryCode").alias("country_code"),
        col("log.number"),
        col("log.duration"),
        col("log.callType").alias("call_type"),
        col("log.callTime").alias("call_time"),
        col("log.callTimeStnd").alias("call_time_stnd"),
        col("log.note").alias("note"),
        col("log.recordingURL").alias("recording_url"),
        col("log.crmStatus").alias("crm_status"),
        col("log.reminderTime").alias("reminder_time"),
        col("log.reminderTimeStnd").alias("reminder_time_stnd"),
        col("log.createdDate").alias("created_date"),
        col("log.modifiedDate").alias("modified_date"),
        col("log.createdDateStnd").alias("created_date_stnd"),
        col("log.modifiedDateStnd").alias("modified_date_stnd"),
        col("log.callTimeStnd").substr(1, 10).alias("call_date"),
        current_timestamp().alias("create_timestamp")
    )
)
flattened_df = flattened_df.withColumn("call_date", to_date("call_date"))
logger.info(f"Total rows to insert: {flattened_df.count()}")

INFO:__main__:Total rows to insert: 92


In [10]:
RDS_HOST, RDS_DBNM, RDS_USER, RDS_PASSWORD, RDS_PORT = get_rds_credentials()

In [11]:
# ---- Step 4: Write to RDS using JDBC ----
flattened_df.write \
    .format("jdbc") \
    .option("url", f'jdbc:postgresql://{RDS_HOST}:{RDS_PORT}/{RDS_DBNM}') \
    .option("dbtable", TABLE_NAME) \
    .option("user", RDS_USER) \
    .option("password", RDS_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

25/06/29 08:00:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [12]:
logger.info("Data written to RDS.")

INFO:__main__:Data written to RDS.


In [13]:
# ---- Step 5: Delete files from S3 ----
for key in files:
    try:
        s3.delete_object(Bucket=BUCKET, Key=key)
        logger.info(f"Deleted: s3://{BUCKET}/{key}")
    except Exception as e:
        logger.warning(f"Failed to delete {key}: {str(e)}")

INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751183724.290796330460005422.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751183730.41258212849521493.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751183731.366016136100991447.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751183733.0923816431047866.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751183740.351797325320594922.txt


...

INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751184008.076262214252498639.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751184009.998559746939981740.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751184010.002992936361830519.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751184017.023796325094506509.txt


INFO:__main__:Deleted: s3://agrim-cdp/data-landing/callyzer/2025-06-29/1751184020.358600932745921728.txt


In [14]:
logger.info("Batch job completed successfully.")

INFO:__main__:Batch job completed successfully.
